# The Goal of this Project is to Predict Whether a person has Diabetes or not

In [ ]:
# Importing the necessary Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Loading the Dataset
df = pd.read_csv("diabetes.csv")

## Exploring the Dataset

In [ ]:
# This tells you how many rows and columns the dataset might contain (r, c)
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
# Returns true for a column having null values, else false
df.isnull().any()

In [ ]:
# Rename DiabetesPedigreeFunction to DPF because thr former is too long
df = df.rename(columns={"DiabetesPedigreeFunction" : "DPF"})
df.head()

In [ ]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

In [ ]:
# Plotting the Outcomes based on the number of dataset entries
plt.figure(figsize=(10,7))
sns.countplot(x="Outcome", data=df)

# Removing the unwanted spines
# plt.gca() stands for "Get Current Axes." this function is handy when it comes to removing the above

plt.gca().spines["top"].set_visible(False)
plt.gca().spines["right"].set_visible(False)

# Headings
plt.xlabel("Has Diabetes")
plt.ylabel("Count")

plt.show()

In the context of Machine Learning and Data Science—specifically when using libraries like Matplotlib or Seaborn—"spines" refer to the lines noting the boundaries of the data area (the top, bottom, left, and right edges of a plot).

When an engineer writes # Removing the unwanted spines, they are usually cleaning up a visualization to make it look more modern, minimalist, or "white-labeled" by hiding the box-like border around a graph.

# Data Cleaning

In [ ]:
# Replacing the 0 values from [ "Glucose". "BLoodPressure". "SkinThickness", "Insulin", "BMI"] by NaN
df_copy = df.copy(deep=True)
# the above as you might know is used to make a deep copy of the original DataFrame. Why "Deep"? In Python/Pandas, a "shallow" copy just points to the same data. If you change df_copy, the original df changes too so to avoid this we need a deep copy.
df_copy[cols] = df_copy[cols].replace(0, np.nan)
df_copy.isnull().sum()

In [ ]:
# To Fill these Nan values the data distribution needs to be understood
# Plotting histogram of dataset before replacing Nan values
df.copy().hist(figsize=(15,15))
plt.show()

In [ ]:
# Replacing Nan value by mean, median depending upon distribution
df_copy["Glucose"].fillna(df_copy["Glucose"].mean(),inplace=True)
df_copy["BloodPressure"].fillna(df_copy["BloodPressure"].mean(), inplace=True)
df_copy["SkinThickness"].fillna(df_copy["SkinThickness"].median(), inplace=True)
df_copy["Insulin"].fillna(df_copy["Insulin"].median(), inplace=True)
df_copy["BMI"].fillna(df_copy["BMI"].median(), inplace=True)

In [ ]:
# Plotting histogram of dataset after replacing NaN values
p = df_copy.hist(figsize=(15,15))

In [ ]:
df_copy.isnull().sum()

# Model Building

In [ ]:
from sklearn.model_selection import train_test_split

X = df_copy.drop(columns="Outcome")
y = df_copy["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train size: {}, X_test size: {}".format(X_train.shape, X_test.shape))

In [ ]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
# Using GridSearchCV to find the best algorithm for this problem
from sklearn.model_selection import GridSearchCV, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [ ]:
# Create a function to calculate best model for this problem
def find_best_model(X, y):
  models = {
    "logistic_regression": {
        "model": LogisticRegression(solver="lbfgs", multi_class="auto"),
        "parameters": {
            "C": [1,5,10]
        }
    },

    "decision_tree": {
        "model": DecisionTreeClassifier(splitter="best"),
        "parameters": {
            "criterion": ["gini", "entropy"],
            "max_depth" : [5, 10]
        }
    },

    "random_forest": {
        "model": RandomForestClassifier(criterion="gini"),
        "parameters": {
            "n_estimators": [10, 15, 20, 50, 100, 200]

        }

    },

    "svm": {
        "model": SVC(gamma="auto"),
        "parameters": {
            "C": [1,10,20],
            "kernel": ["rbf", "linear"]
        }
    }
  }
  scores = []
  cv_shuffle = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

  for model_name, model_params in models.items():
    gs = GridSearchCV(model_params["model"], model_params["parameters"], cv = cv_shuffle, return_train_score=False)
    gs.fit(X, y)
    scores.append({
        "model": model_name,
        "best_parameters": gs.best_params_,
        "score": gs.best_score_
    })

  return pd.DataFrame(scores, columns=["model", "best_parameters", "score"])
find_best_model(X_train, y_train)


Note: Since the Random Forest algorithm has the highest accuracy, we further finetune the model using hyperparameter Optimization

In [ ]:
# Using cross_val_score for gaining average accuracy
from sklearn.model_selection import cross_val_score
scores = cross_val_score(RandomForestClassifier(n_estimators=20, random_state=42), X_train, y_train, cv= 5)
print("Average Accuracy: {}%".format(round(sum(scores)*100/len(scores)), 3))


In [ ]:
# Creating Random Forest Model
classifier = RandomForestClassifier(n_estimators=20, random_state=42)
classifier.fit(X_train, y_train)

# Model Evaluation

In [ ]:
# Create a confusion matrix
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Accuracy Score
score = round(accuracy_score(y_test, y_pred),4)*100
print("Accuracy on test set: {}%".format(score))

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred))

#Predictions

In [ ]:
# Creating a function for prediction
def predict_diabetes(Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age):
  preg = int(Pregnancies)
  glucose = float(Glucose)
  bp = float(BloodPressure)
  st = float(SkinThickness)
  insulin = float(Insulin)
  bmi = float(BMI)
  dpf = float(DPF)
  age = int(Age)

  x = [[preg, glucose, bp, st, insulin, bmi, dpf, age]]
  x = sc.transform(x)

  return classifier.predict(x)

In [ ]:
# Prediction 1
# Input sequence: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age
prediction = predict_diabetes(2, 81, 72, 15, 76, 30.1, 0.547, 25)[0]
if prediction:
  print("Oops! You have diabetes.")

else:
  print("Great! You don't have diabetes.")


In [ ]:
# Prediction 2
# Input sequence: Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DPF, Age
prediction = predict_diabetes(1, 117, 88, 24, 145, 34.5, 0.403, 40)[0]
if prediction:
  print("Oops! You have diabetes.")

else:
  print("Great! You don't have diabetes.")
